# 🏷️ Tạo nhãn phân loại: `IndusLabel` · `ProductLabel` · `ProductBrandLabel`

**Nguồn:** `2025.xlsx` + `2026.xlsx` — Sheet `Table1`

| Label | Ý nghĩa | Nguồn gốc |
|-------|---------|-----------|
| `IndusLabel` | Ngành kinh doanh của cửa hàng | Map từ `Loại địa điểm` → 15 ngành macro |
| `ProductLabel` | Danh mục sản phẩm | Rule-based theo từ khóa trong `ItemName` |
| `ProductBrandLabel` | Thương hiệu sản phẩm | Rule-based theo tên thương hiệu trong `ItemName` |

---


## ⚙️ 0. Cấu hình

In [ ]:
FILE_2025   = "2025.xlsx"
FILE_2026   = "2026.xlsx"
SHEET       = "Table1"
OUTPUT_FILE = "labeled_items.csv"

print("✅ Cấu hình sẵn sàng")


## 📦 1. Import & đọc dữ liệu

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings("ignore")

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

# Đọc & gộp
df25 = pd.read_excel(FILE_2025, sheet_name=SHEET, dtype=str)
df26 = pd.read_excel(FILE_2026, sheet_name=SHEET, dtype=str)
df   = pd.concat([df25, df26], ignore_index=True)

# Tách header hóa đơn vs dòng sản phẩm
df_bills = df[df["SanPham"].isna()].copy()
df_items = df[df["SanPham"].notna()].copy()

# Merge thông tin từ bill header vào item
bill_meta = df_bills[["Bill ID","Loại địa điểm","Địa điểm",
                       "Khu vực","User ID","Ngày","Tình trạng"]].copy()
df_items  = df_items.merge(bill_meta, on="Bill ID", how="left",
                           suffixes=("","_bill"))

# Parse ItemName từ cột SanPham (format: TênSP`` SLg`` Giá)
df_items["ItemName"]  = df_items["SanPham"].str.split("``").str[0].str.strip()
df_items["SoLuong"]   = pd.to_numeric(df_items["SoLuong"], errors="coerce")
df_items["Gia"]       = pd.to_numeric(df_items["Gia"],     errors="coerce")

print(f"Dataset gộp      : {len(df):,} dòng")
print(f"Dòng header bill : {len(df_bills):,}")
print(f"Dòng sản phẩm   : {len(df_items):,}")
print(f"ItemName unique  : {df_items['ItemName'].nunique():,}")


In [ ]:
display(df_items[["Bill ID","ItemName","SoLuong","Gia","Loại địa điểm","Địa điểm"]].head(8))

---
## 🏭 2. `IndusLabel` — Ngành kinh doanh của địa điểm

**Logic:** Map trực tiếp từ `Loại địa điểm` (39 loại) → 15 ngành macro.  
Các hóa đơn không có `Loại địa điểm` (null ~48%) → `Chưa phân loại`.


In [ ]:
INDUSTRY_MAP = {
    # ── Bán lẻ ──────────────────────────────────────────────────────
    "Siêu thị"                                   : "Bán lẻ & Siêu thị",
    "Tạp hóa"                                    : "Bán lẻ & Siêu thị",
    "Cửa hàng đặc sản vùng miền"                 : "Bán lẻ & Siêu thị",
    # ── F&B ─────────────────────────────────────────────────────────
    "Coffee"                                     : "F&B – Cà phê",
    "Trà sữa"                                    : "F&B – Trà sữa",
    "Nhà hàng"                                   : "F&B – Nhà hàng",
    "Quán ăn"                                    : "F&B – Quán ăn",
    "Quán đồ ăn vặt"                             : "F&B – Đồ ăn vặt",
    "Tiệm bánh"                                  : "F&B – Tiệm bánh",
    "Quán nước ép và sinh tố"                    : "F&B – Nước ép & Sinh tố",
    "Quán nhậu"                                  : "F&B – Quán nhậu",
    "Quán bar, pub, lounge"                      : "F&B – Bar & Pub",
    # ── Y tế & Làm đẹp ──────────────────────────────────────────────
    "Nhà thuốc"                                  : "Y tế & Dược phẩm",
    "Trung tâm spa và chăm sóc sức khỏe"         : "Y tế & Làm đẹp",
    "Tiệm nail"                                  : "Y tế & Làm đẹp",
    "Tiệm làm tóc (salon tóc)"                  : "Y tế & Làm đẹp",
    "Mỹ phẩm"                                   : "Y tế & Làm đẹp",
    # ── Thời trang ──────────────────────────────────────────────────
    "Cửa hàng thời trang"                        : "Thời trang & Phụ kiện",
    "Cửa hàng đồ thể thao"                      : "Thời trang & Phụ kiện",
    # ── Giải trí ─────────────────────────────────────────────────────
    "Rạp phim"                                   : "Giải trí",
    "Tổ hợp vui chơi giải trí"                  : "Giải trí",
    "Karaoke"                                    : "Giải trí",
    "Escape room và board game café"             : "Giải trí",
    "Show diễn nghệ thuật"                       : "Giải trí",
    # ── Giáo dục ─────────────────────────────────────────────────────
    "Hiệu sách"                                  : "Giáo dục & Sách",
    "Trung tâm ngoại ngữ"                        : "Giáo dục & Sách",
    # ── Công nghệ ────────────────────────────────────────────────────
    "Cửa hàng điện thoại và phụ kiện công nghệ" : "Công nghệ & Điện máy",
    "Cửa hàng điện máy"                         : "Công nghệ & Điện máy",
    # ── Gia dụng & Trẻ em ────────────────────────────────────────────
    "Cửa hàng đồ gia dụng"                      : "Đồ gia dụng",
    "Cửa hàng đồ chơi trẻ em"                   : "Đồ chơi & Trẻ em",
    # ── Dịch vụ & Du lịch ────────────────────────────────────────────
    "Dịch vụ vận chuyển và giao hàng nhanh"     : "Dịch vụ & Khác",
    "Dịch vụ tổ chức sự kiện và trang trí tiệc" : "Dịch vụ & Khác",
    "Tiệm sửa xe máy, ô tô"                     : "Dịch vụ & Khác",
    "Studio chụp ảnh"                            : "Dịch vụ & Khác",
    "Tòa nhà, văn phòng, công ty"               : "Dịch vụ & Khác",
    "Cửa hàng hoa tươi và dịch vụ trang trí hoa": "Dịch vụ & Khác",
    "Khu nghỉ dưỡng (Resort)"                   : "Du lịch & Lưu trú",
    "Khách sạn"                                 : "Du lịch & Lưu trú",
    "Khác"                                      : "Khác",
}

df_items["IndusLabel"] = (
    df_items["Loại địa điểm"]
    .map(INDUSTRY_MAP)
    .fillna("Chưa phân loại")
)

n_labeled = df_items["IndusLabel"].ne("Chưa phân loại").sum()
print(f"IndusLabel coverage: {n_labeled:,} / {len(df_items):,} ({n_labeled/len(df_items)*100:.1f}%)")
print("\nPhân phối IndusLabel:")
display(df_items["IndusLabel"].value_counts().reset_index()
        .rename(columns={"count":"Số dòng"}))


In [ ]:
vc = df_items["IndusLabel"].value_counts()
fig, ax = plt.subplots(figsize=(11, 6))
colors  = plt.cm.tab20.colors[:len(vc)]
bars    = ax.barh(vc.index[::-1], vc.values[::-1], color=colors[::-1], edgecolor="white")
for bar, val in zip(bars, vc.values[::-1]):
    ax.text(bar.get_width() + 50, bar.get_y() + bar.get_height()/2,
            f"{val:,}", va="center", fontsize=9)
ax.set_xlabel("Số dòng sản phẩm")
ax.set_title("Phân phối IndusLabel")
plt.tight_layout(); plt.show()


---
## 🛒 3. `ProductLabel` — Danh mục sản phẩm

**Logic:** Rule-based theo từ khóa trong `ItemName` (uppercase).  
Ưu tiên từ trên xuống dưới — danh mục nào match trước thì dùng.


In [ ]:
def classify_product(name):
    """Phân loại sản phẩm theo tên. Trả về ProductLabel (str)."""
    if pd.isna(name) or str(name).strip() == "":
        return "Không xác định"
    n = str(name).upper().strip()

    # ── Dịch vụ & phụ phí (ưu tiên đầu) ──────────────────────────
    svc_kw = ["SERVICE CHARGE","KHĂN LẠNH","KHAN LANH","GIO CHOI BI A",
              "VE TRONG TUAN","BAO LI XI","TIET KIEM","DISCOUNT","COUPON",
              "ECODE","GIAM GIA","TOPPING","EXTRA ","ADD-ON",
              "SPOON","FORK","PERSONAL CUP","CUP ","VE PHU THU",
              "VE DI XE","KHUYẾN MÃI","TANG KEM","BAO IN ","BAO NHUA",
              "SERVICE ","DICH VU ","COMBO ","SET BO"]
    if any(kw in n for kw in svc_kw):
        return "Dịch vụ & Phụ phí"

    # ── Vé & Giải trí ────────────────────────────────────────────
    ticket_kw = ["VE PHIM","VÉ PHIM","VE DI XE","VÉ ĐI XE BUYT",
                 "BUFFET ","SUẤT CHIẾU","SUAT CHIEU","HOAT HUYET",
                 "VE TUAN","GIO CHOI"]
    if any(kw in n for kw in ticket_kw):
        return "Vé & Giải trí"

    # ── Đồ uống có cồn ────────────────────────────────────────────
    alc_kw = ["BIA ","BEER","HEINEKEN","TIGER","BUDWEISER","CORONA ",
              "333 ","BIA 333","RUOU ","RƯỢU","SOJU","WINE ","VANG ",
              "WHISKY","VODKA","GIN "]
    if any(kw in n for kw in alc_kw):
        return "Đồ uống có cồn"

    # ── Cà phê & Trà (đặt trước nước giải khát) ──────────────────
    coffee_kw = ["CA PHE","CÀ PHÊ","CAFE ","CAPHE","ESPRESSO","LATTE",
                 "AMERICANO","CAPPUCCINO","COLD BREW","DRIP ","PHIN ",
                 "BAC XIU","BẠC XỈU","MACCHIATO","MOCHA","CJOY",
                 "TRA SUA","TRÀ SỮA","MILK TEA","BUBBLE TEA",
                 "TRAN CHAU","TRÂN CHÂU","TRA THACH","TRÀ THẠCH",
                 "TRA THANH","TRÀ THANH","TRA NGOC TRAI","TRA SEN VANG",
                 "TRA LUCKY","TRA PHUC LONG","TRA DAO","HOT COFFEE",
                 "MATCHA","OLONG ","OOLONG","HI-TEA","HITEA",
                 "SMOOTHIE","FREEZE ","PHINDI","CACAO",
                 "BAC XIEU","BẠC XIU","TRÀ ĐÁ","TRA DA "]
    if any(kw in n for kw in coffee_kw):
        return "Cà phê & Trà"

    # ── Nước giải khát ────────────────────────────────────────────
    soft_kw = ["COCA","PEPSI","SPRITE","7UP","FANTA","MIRINDA",
               "STING ","REVIVE","AQUAFINA","LAVIE","DASANI","EVIAN ",
               "NUOC TINH","NƯỚC TINH","NUOC KHOANG","NUOC SUOI",
               "NƯỚC SUỐI","RED BULL","REDBULL","MONSTER ","HELL ",
               "POCARI","ION SUPPLY","C2 ","TREETOP","NUOC EP",
               "NUTRIBOOST","NUTRI BOOST","REAL LEAF","NESTEA","LIPTON",
               "NƯỚC ÉP","SINH TỐ","NƯỚC DỪA","COCONUT WATER",
               "COCOXIM","PEPSI ","COCA ","7-UP","SPRITE "]
    if any(kw in n for kw in soft_kw):
        return "Nước giải khát"

    # ── Sữa & Dinh dưỡng ─────────────────────────────────────────
    dairy_kw = ["SUA ","SỮA ","MILK","VINAMILK","TH TRUE","DUTCH LADY",
                "MILO ","ANLENE","SIMILAC","APTAMIL","NUTIFOOD","NAN ",
                "FRISO","ENFAMIL","MEIJI","ABBOTT","PEDIASURE",
                "DIELAC","NESTLE ","NESTLÉ","VINAMILK "]
    if any(kw in n for kw in dairy_kw):
        return "Sữa & Dinh dưỡng"

    # ── Bánh kẹo & Snack ──────────────────────────────────────────
    snack_kw = ["BANH ","BÁNH ","SNACK","KHOAI TAY CHIEN","KHOAI TÂY CHIÊN",
                "CHIP ","KEO ","KẸO ","CHOCOLATE","SOCOLA ","OREO","KITKAT",
                "KIT KAT","POCKY","GUMMY","GUMI ","JELLY ","WAFER",
                "COOKIE","BISKUAT","COSY ","SOLITE","YOUME","GOUTE",
                "DANISA","DELORI","MUFFIN","DONUT","CUPCAKE",
                "CROISSANT","BAGUETE","MOI HINH POKEMON",
                "THE PACK CARD","REECEN","KEO ME"]
    if any(kw in n for kw in snack_kw):
        return "Bánh kẹo & Snack"

    # ── Mì & Ăn liền ─────────────────────────────────────────────
    instant_kw = ["MI ","MÌ ","AN LIEN","CHAO ","PHO ","PHỞ ",
                  "HU TIEU","HỦ TIẾU","BUN ","BÚN ","BANH CANH",
                  "MI KIM CHI","MI TRON","MI LAU","MI XAO",
                  "UDON","RAMEN","MIEN ","MIẾN "]
    if any(kw in n for kw in instant_kw):
        return "Mì & Ăn liền"

    # ── Thực phẩm tươi sống ───────────────────────────────────────
    fresh_kw = ["THIT ","THỊT ","CA MU","CÁ ","CA ","TOM ","TÔM ",
                "CUA ","CHEM CHEP","NGAO ","NGHEU ","BO HOC","XUONG ",
                "XƯƠNG ","GA ","GÀ ","VIT ","VỊT ","HAI SAN ","HẢI SẢN",
                "HEO ","LONG HEO","TUOI SONG","TƯƠI SỐNG",
                "RAU ","CÀI ","CAI ","BROKOLI","SALAD RAU",
                "DUA LEO","CA ROT","HANH ","KHOAI ","TOMATO",
                "CAM VAT","CHUOI ","TRAI CAY","TRÁI CÂY","BUOI ",
                "CHERRY ","NHAN ","THANH LONG","XOAI ","DURIAN",
                "DAU TU","DAU XANH","NAM ","NẤM ","THIT XAY",
                "THỊT NẠC","BO KHO","TIET CANH","TIẾT CANH"]
    if any(kw in n for kw in fresh_kw):
        return "Thực phẩm tươi sống"

    # ── Thực phẩm chế biến ────────────────────────────────────────
    food_kw = ["XUC XICH","XÚC XÍCH","HOT DOG","HAM ","LAP XUONG",
               "PATE ","NEM ","CHA ","GIO ","GIÒ ","COM ","CƠM ",
               "YAOURT","SUA CHUA","SỮA CHUA","KIM CHI","DUA MUOI",
               "MAYONNAISE","SOT ","SỐT ","DAU TUONG","DAU HU",
               "TRUNG ","TRỨNG ","CA HOP","CA TRICH","CA THU",
               "MEAT DELI","VISSAN","BASA ","XUC XICH","HOTDOG",
               "XUCXICH","TOKBOKKI","BANH XEO","BANH TRANG",
               "CHA GIO","SPRING ROLL","DUMPLING","HA CAO",
               "CRISY ","CRISPY "]
    if any(kw in n for kw in food_kw):
        return "Thực phẩm chế biến"

    # ── Gia vị & Nguyên liệu ─────────────────────────────────────
    condiment_kw = ["GIA VI","GIA VỊ","NUOC MAM","NƯỚC MẮM","NUOC TUONG",
                    "TINH BOT","BOT ","COOKING","KNORR","MAGGI",
                    "AJINOMOTO","CHINSU","DAU AN","DẦU ĂN","MUOI ",
                    "MUỐI ","DUONG ","ĐƯỜNG ","NAM NGAN"]
    if any(kw in n for kw in condiment_kw):
        return "Gia vị & Nguyên liệu"

    # ── Thuốc & Dược phẩm ────────────────────────────────────────
    pharma_kw = ["THUOC ","THUỐC ","VITAMIN","OMEGA ","COLLAGEN",
                 "SUPPLEMENT","CAPSULE","TABLET","VIEN ","VIÊN ",
                 "SYRUP","SYP ","INJECTION","KIM TIEM","BONG GIAC",
                 "BANG KEO Y TE","NUOC MUOI","NASAL",
                 "PARACETAMOL","IBUPROFEN","AMOXICILLIN","OMEPRAZOLE",
                 "NATRI CLORID","NUOC OXY GIA","SYRINGE","BOM TIEM",
                 "HOAT HUYET"]
    if any(kw in n for kw in pharma_kw):
        return "Thuốc & Dược phẩm"

    # ── Mỹ phẩm & CSKN ───────────────────────────────────────────
    beauty_kw = ["DAU GOI","DẦU GỘI","SUA TAM","SỮA TẮM","TOOTHPASTE",
                 "COLGATE","SENSODYNE","SHAMPOO","CONDITIONER",
                 "NUOC HOA","NƯỚC HOA","SON ","PHAN ","PHẤN ",
                 "KEM DUONG","LOTION","SERUM ","TONER","NUOC TAY TRANG",
                 "MASK ","MAT NA","GILLETTE","DAO CAO","HEAD ","DOVE ",
                 "CLEAR ","SUNSILK","PANTENE","GARNIER","NIVEA",
                 "BIODERMA","NEUTROGENA","CERAVE","THE ORDINARY",
                 "TINH DAU DUONG TOC","KARANZ","KEM DANH RANG"]
    if any(kw in n for kw in beauty_kw):
        return "Mỹ phẩm & Chăm sóc cá nhân"

    # ── Thời trang & Phụ kiện ────────────────────────────────────
    fashion_kw = ["AO ","ÁO ","QUAN ","QUẦN ","GIAY ","GIÀY ","DEP ",
                  "DÉP ","TUI ","TÚI ","VONG ","VÒNG ","NHAN ",
                  "DAY CHUYEN","THAT LUNG","THẮT LƯNG","JEAN ","JEANS",
                  "SKIRT","DRESS","SHIRT","JACKET","HOODIE","TANKTOP",
                  "CROP ","BLAZER","TSHIRT","T-SHIRT"]
    if any(kw in n for kw in fashion_kw):
        return "Thời trang & Phụ kiện"

    # ── Đồ gia dụng & Vệ sinh ────────────────────────────────────
    household_kw = ["NUOC RUA CHEN","NUOC LAU SAN","KHU KHUAN",
                    "GIAY LAU","GIAY THAM","GIAY TOILET","KHAN GIAY",
                    "KHĂN GIẤY","OMO ","ARIEL ","TIDE ","COMFORT",
                    "DOWNY","SUNLIGHT","VIM ","DAO ROC GIAY","BANG DINH"]
    if any(kw in n for kw in household_kw):
        return "Đồ gia dụng & Vệ sinh"

    # ── Điện tử & Công nghệ ──────────────────────────────────────
    tech_kw = ["IPHONE","SAMSUNG","XIAOMI","OPPO","VIVO","REALME",
               "LAPTOP","MACBOOK","IPAD","TABLET","EARPHONE","TAI NGHE",
               "CHARGER","CAP USB","CABLE ","KEYBOARD","CHUOT ",
               "MOUSE ","HDMI","SIM ","THE NHO"]
    if any(kw in n for kw in tech_kw):
        return "Điện tử & Công nghệ"

    # ── Văn phòng phẩm & Sách ────────────────────────────────────
    stationery_kw = ["GIAY A4","GIẤY A4","BUT ","BÚT ","SACH ","SÁCH ",
                     "NOTEBOOK","VPGP","VPP ","HOP BI MAT","BUT LONG",
                     "RULER","THUOC KE","SCISSORS","KÉO ","STAPLER",
                     "TIENG ANH","STUDENT BOOK","TAC PHAM ","LUYEN THI"]
    if any(kw in n for kw in stationery_kw):
        return "Văn phòng phẩm & Sách"

    return "Khác"

df_items["ProductLabel"] = df_items["ItemName"].apply(classify_product)

n_cov = df_items["ProductLabel"].ne("Khác").sum()
print(f"ProductLabel coverage: {n_cov:,} / {len(df_items):,} ({n_cov/len(df_items)*100:.1f}%)")
print("\nPhân phối ProductLabel:")
display(df_items["ProductLabel"].value_counts().reset_index()
        .rename(columns={"count":"Số dòng"}))


In [ ]:
vc2 = df_items["ProductLabel"].value_counts()
fig, ax = plt.subplots(figsize=(11, 7))
cmap    = plt.cm.tab20.colors
bars    = ax.barh(vc2.index[::-1], vc2.values[::-1],
                  color=cmap[:len(vc2)][::-1], edgecolor="white")
for bar, val in zip(bars, vc2.values[::-1]):
    ax.text(bar.get_width() + 30, bar.get_y() + bar.get_height()/2,
            f"{val:,}", va="center", fontsize=9)
ax.set_xlabel("Số dòng sản phẩm")
ax.set_title("Phân phối ProductLabel")
plt.tight_layout(); plt.show()


---
## 🏷️ 4. `ProductBrandLabel` — Thương hiệu sản phẩm

**Logic:** Dò từ khóa tên thương hiệu trong `ItemName` (uppercase, theo thứ tự ưu tiên).  
Sản phẩm không nhận diện được thương hiệu → `Không rõ nhãn`.


In [ ]:
# Danh sách (list_keywords, brand_name) — ưu tiên từ trên xuống
BRAND_RULES = [
    # ── Nước giải khát ─────────────────────────────────────────────
    (["COCA COLA","COCA-COLA","COCACOLA","COCA ","COCA\n","COCA$"], "Coca-Cola"),
    (["PEPSI"],                                                        "Pepsi"),
    (["SPRITE"],                                                       "Sprite"),
    (["7UP","7 UP","7-UP"],                                           "7UP"),
    (["FANTA"],                                                       "Fanta"),
    (["STING ","STING\n"],                                           "Sting"),
    (["REVIVE"],                                                      "Revive"),
    (["AQUAFINA"],                                                    "Aquafina"),
    (["DASANI"],                                                      "Dasani"),
    (["LAVIE"],                                                       "Lavie"),
    (["RED BULL","REDBULL"],                                          "Red Bull"),
    (["MONSTER "],                                                    "Monster Energy"),
    (["POCARI"],                                                      "Pocari Sweat"),
    (["ION SUPPLY"],                                                  "Ion Supply"),
    (["NUTRIBOOST","NUTRI BOOST"],                                    "Nutriboost"),
    (["REAL LEAF"],                                                   "Real Leaf"),
    (["NESTEA"],                                                      "Nestea"),
    (["LIPTON"],                                                      "Lipton"),
    (["C2 ","C2\n"],                                                "C2"),
    (["COCOXIM"],                                                     "Cocoxim"),
    (["HTH "],                                                        "HTH"),
    # ── Bia & Đồ uống có cồn ─────────────────────────────────────
    (["TIGER ","BIA TIGER","TIGER BEER"],                            "Tiger Beer"),
    (["HEINEKEN"],                                                    "Heineken"),
    (["BUDWEISER"],                                                   "Budweiser"),
    (["333 ","BIA 333","BIA 333\n"],                               "Bia 333"),
    (["SAPPORO"],                                                     "Sapporo"),
    (["SOJU","JINRO","TAE YANG"],                                    "Soju"),
    (["CORONA "],                                                     "Corona"),
    # ── Cà phê & Trà thương hiệu ─────────────────────────────────
    (["HIGHLANDS","HIGHLAND COFFEE"],                                 "Highlands Coffee"),
    (["PHUC LONG","PHÚC LONG"],                                      "Phúc Long Tea"),
    # ── Sữa & Dinh dưỡng ─────────────────────────────────────────
    (["VINAMILK"],                                                    "Vinamilk"),
    (["TH TRUE","TH TRUEMILK","TH TRUE MILK"],                      "TH True Milk"),
    (["DUTCH LADY"],                                                  "Dutch Lady"),
    (["MILO ","MILO\n","MILO$"],                                    "Milo"),
    (["NESTLÉ","NESTLE "],                                           "Nestlé"),
    (["ANLENE"],                                                      "Anlene"),
    (["ENSURE"],                                                      "Ensure (Abbott)"),
    (["SIMILAC"],                                                     "Similac"),
    (["APTAMIL"],                                                     "Aptamil"),
    (["NUTIFOOD"],                                                    "Nutifood"),
    (["DIELAC"],                                                      "Dielac"),
    (["FRISO"],                                                       "Friso"),
    (["MEIJI"],                                                       "Meiji"),
    # ── Bánh kẹo & Snack ─────────────────────────────────────────
    (["OISHI"],                                                       "Oishi"),
    (["POCA ","POCA\n","LAY'S","LAYS "],                           "Poca / Lay's"),
    (["OREO"],                                                        "Oreo"),
    (["KITKAT","KIT KAT"],                                           "KitKat"),
    (["POCKY"],                                                       "Pocky"),
    (["DANISA"],                                                      "Danisa"),
    (["COSY "],                                                       "Cosy"),
    (["GOUTE"],                                                       "Goute"),
    (["SOLITE"],                                                      "Solite"),
    (["BISKUAT"],                                                     "Biskuat"),
    (["REECEN"],                                                      "Reecen"),
    (["HIGLANDS COFFEE Cafe sua"],                                    "Highlands Coffee (lon)"),
    # ── Thực phẩm chế biến ────────────────────────────────────────
    (["MEAT DELI"],                                                   "Meat Deli"),
    (["VISSAN"],                                                      "Vissan"),
    (["CHINSU"],                                                      "Chin-su"),
    (["BIBIGO"],                                                      "Bibigo"),
    (["CAMPINA"],                                                     "Campina"),
    (["WM_","WINMART"],                                              "WinMart"),
    (["C - ","COOPMART","COOP ","C-"],                              "Co.op Mart"),
    (["MODERN "],                                                     "Modern"),
    # ── Gia vị ────────────────────────────────────────────────────
    (["MAGGI"],                                                       "Maggi"),
    (["KNORR"],                                                       "Knorr"),
    (["AJINOMOTO"],                                                   "Ajinomoto"),
    # ── Chăm sóc cá nhân ─────────────────────────────────────────
    (["COLGATE"],                                                     "Colgate"),
    (["SENSODYNE"],                                                   "Sensodyne"),
    (["GILLETTE"],                                                    "Gillette"),
    (["HEAD "],                                                       "Head & Shoulders"),
    (["DOVE "],                                                       "Dove"),
    (["CLEAR "],                                                      "Clear"),
    (["SUNSILK"],                                                     "Sunsilk"),
    (["PANTENE"],                                                     "Pantene"),
    (["GARNIER"],                                                     "Garnier"),
    (["NIVEA"],                                                       "Nivea"),
    (["BIODERMA"],                                                    "Bioderma"),
    (["NEUTROGENA"],                                                  "Neutrogena"),
    (["CERAVE"],                                                      "CeraVe"),
    (["KARANZ"],                                                      "Karanz"),
    # ── Giặt giũ & Gia dụng ──────────────────────────────────────
    (["OMO "],                                                        "OMO"),
    (["ARIEL "],                                                      "Ariel"),
    (["DOWNY"],                                                       "Downy"),
    (["COMFORT"],                                                     "Comfort"),
    (["SUNLIGHT"],                                                    "Sunlight"),
    # ── VPP ──────────────────────────────────────────────────────
    (["THIEN LONG","THIÊN LONG"],                                    "Thiên Long"),
]

def get_brand(name):
    """Trả về thương hiệu nhận diện được, hoặc 'Không rõ nhãn'."""
    if pd.isna(name):
        return "Không rõ nhãn"
    n = str(name).upper().strip()
    for keywords, brand in BRAND_RULES:
        if any(kw.upper() in n for kw in keywords):
            return brand
    return "Không rõ nhãn"

df_items["ProductBrandLabel"] = df_items["ItemName"].apply(get_brand)

known = df_items["ProductBrandLabel"].ne("Không rõ nhãn").sum()
print(f"ProductBrandLabel coverage: {known:,} / {len(df_items):,} ({known/len(df_items)*100:.1f}%)")
print("\nPhân phối top 30 thương hiệu:")
display(df_items[df_items["ProductBrandLabel"] != "Không rõ nhãn"]
        ["ProductBrandLabel"].value_counts().head(30)
        .reset_index().rename(columns={"count":"Số dòng"}))


In [ ]:
top_brands = (df_items[df_items["ProductBrandLabel"] != "Không rõ nhãn"]
              ["ProductBrandLabel"].value_counts().head(25))

fig, ax = plt.subplots(figsize=(10, 7))
colors  = plt.cm.tab20.colors
bars    = ax.barh(top_brands.index[::-1], top_brands.values[::-1],
                  color=colors[:len(top_brands)][::-1], edgecolor="white")
for bar, val in zip(bars, top_brands.values[::-1]):
    ax.text(bar.get_width() + 2, bar.get_y() + bar.get_height()/2,
            f"{val:,}", va="center", fontsize=9)
ax.set_xlabel("Số dòng sản phẩm")
ax.set_title("Top 25 ProductBrandLabel (không tính 'Không rõ nhãn')")
plt.tight_layout(); plt.show()


---
## 📊 5. Phân tích chéo & Cross-validation

In [ ]:
# IndusLabel vs ProductLabel — crosstab
ct = pd.crosstab(df_items["IndusLabel"], df_items["ProductLabel"])
print("Cross-tab IndusLabel × ProductLabel (top combination):")
ct_long = ct.stack().reset_index()
ct_long.columns = ["IndusLabel","ProductLabel","Count"]
ct_long = ct_long[ct_long["Count"] > 0].sort_values("Count", ascending=False)
display(ct_long.head(20))


In [ ]:
# Kiểm tra: Sản phẩm còn trong 'Khác' — xem mẫu để cải thiện rule
others = df_items[df_items["ProductLabel"] == "Khác"]["ItemName"].dropna()
print(f"Sản phẩm chưa phân loại ('Khác'): {len(others):,}")
print("\nMẫu 30 ItemName chưa phân loại (gợi ý bổ sung rule):")
for name in others.sample(min(30, len(others)), random_state=42).tolist():
    print(f"  {name[:90]}")


In [ ]:
# Thương hiệu theo ngành
brand_by_indus = (
    df_items[df_items["ProductBrandLabel"] != "Không rõ nhãn"]
    .groupby(["IndusLabel","ProductBrandLabel"])
    .size().reset_index(name="Count")
    .sort_values(["IndusLabel","Count"], ascending=[True,False])
)
print("Top brand theo từng ngành:")
display(brand_by_indus.groupby("IndusLabel").head(3))


## 💾 6. Export kết quả

In [ ]:
# Chọn cột output
output_cols = [
    "Bill ID", "User ID", "Ngày", "Địa điểm", "Loại địa điểm",
    "Khu vực", "Tình trạng", "Tổng đơn hàng",
    "ItemName", "SoLuong", "Gia",
    "IndusLabel", "ProductLabel", "ProductBrandLabel",
]
df_out = df_items[[c for c in output_cols if c in df_items.columns]].copy()

df_out.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
print(f"✅ Đã xuất: {OUTPUT_FILE}  ({len(df_out):,} dòng × {len(df_out.columns)} cột)")
print("\nXem trước:")
display(df_out.head(10))


In [ ]:
# Tổng kết
print("=" * 60)
print("  TỔNG KẾT 3 LABEL")
print("=" * 60)
for label_col, fallback in [
    ("IndusLabel",         "Chưa phân loại"),
    ("ProductLabel",       "Khác"),
    ("ProductBrandLabel",  "Không rõ nhãn"),
]:
    n_total   = len(df_items)
    n_labeled = df_items[label_col].ne(fallback).sum()
    n_cats    = df_items[label_col].nunique()
    print(f"\n  {label_col}")
    print(f"    Coverage  : {n_labeled:,}/{n_total:,} ({n_labeled/n_total*100:.1f}%)")
    print(f"    Số nhãn   : {n_cats}")
    print(f"    Top 3     : {df_items[label_col].value_counts().head(3).index.tolist()}")


---
## 💡 Lưu ý & Cải thiện

### Coverage hiện tại
| Label | Coverage | Giới hạn |
|-------|----------|---------|
| `IndusLabel` | ~47% | 48% hóa đơn không có `Loại địa điểm` (null) |
| `ProductLabel` | ~65% | 35% còn nhãn `Khác` — tên SP tự do, viết tắt, tiếng Việt không dấu |
| `ProductBrandLabel` | ~8% | Chỉ nhận diện sản phẩm có thương hiệu rõ ràng |

### Cách cải thiện
1. **IndusLabel:** Bổ sung phân loại dựa trên `Địa điểm` (tên cửa hàng) cho các bill không có `Loại địa điểm`
2. **ProductLabel:** Thêm rule cho từ khóa thường thấy trong mục `Khác` (xem cell trên để lấy mẫu)
3. **ProductBrandLabel:** Mở rộng `BRAND_RULES` theo danh sách từ cell "Mẫu brand" bên trên
4. **ML approach:** Dùng `ProductLabel` đã gán làm training data để train classifier cho phần `Khác`
